In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer
import yfinance as yf
import random
from dataclasses import dataclass
from typing import Dict, Tuple
from tqdm import tqdm
import itertools

In [ ]:

# =============================================================================
# Step 1. Prediction Model: z -> r_hat
# =============================================================================

class PredictionModel(nn.Module):
    """
    예측 모델 f_theta(z) -> r_hat

    Args:
        input_dim  : 입력 피처 차원
        hidden_dim : 히든 레이어 차원
        N          : 예측 기간 (time steps)
        m          : 자산 수
    
    Output:
        r_hat : (batch, N, m) - 각 자산의 기간별 예측 수익률
    """
    def __init__(self, input_dim: int, hidden_dim: int, N: int, m: int):
        super().__init__()
        self.N = N
        self.m = m
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, N * m),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z : (batch, input_dim)
        Returns:
            r_hat : (batch, N, m)
        """
        batch = z.shape[0]
        out = self.net(z)               # (batch, N*m)
        r_hat = out.view(batch, self.N, self.m)
        return r_hat


# =============================================================================
# Step 2. Cumulative Return Path: r_hat -> y_hat
# =============================================================================

def compute_cumulative_path(r: torch.Tensor) -> torch.Tensor:
    """
    r_hat에서 누적 수익률 경로 y_hat을 생성

    y^j(t) = sum_{k=1}^{t} r^j(k)

    Args:
        r : (batch, N, m) - 기간별 수익률
    Returns:
        y : (batch, N, m) - 시점별 누적 수익률
    """
    y = torch.cumsum(r, dim=1)  # t축으로 누적합
    return y


# =============================================================================
# Step 3. Optimization Layer: y_hat -> x_star  (Formulation 16)
# =============================================================================

def build_optimization_layer(N: int, m: int) -> CvxpyLayer:
    """
    Formulation (16)을 cvxpylayers로 구성

    목적함수:
        max  (1/dC) * y_hat(N)^T x
        (dC는 상수이므로 y_hat(N)^T x 최대화와 동치)

    제약식:
        u_k - y_hat_k^T x <= n1 * C,    k = 1,...,N   (drawdown 상한)
        u_k >= y_hat_k^T x,              k = 1,...,N   (running max >= 현재값)
        u_k >= u_{k-1},                  k = 1,...,N   (running max 단조증가)
        u_0 = 0
        x_min <= x_i <= x_max,          i = 1,...,m   (box constraint)

    Parameters (cvxpy):
        Y_hat : (N, m) - 예측 누적 수익률 경로 (각 행이 y_hat(t))
        n1C   : scalar - 허용 최대 drawdown 한도 (n1 * C)
        x_min : scalar
        x_max : scalar

    Returns:
        CvxpyLayer - differentiable optimization layer
    """
    # --- cvxpy 변수 ---
    x = cp.Variable(m, name="x")           # 포트폴리오 가중치 (m,)
    u = cp.Variable(N + 1, name="u")       # running max 보조변수 (N+1,) : u[0],...,u[N]

    # --- cvxpy 파라미터 ---
    Y_hat = cp.Parameter((N, m), name="Y_hat")   # 예측 누적 수익률 경로
    n1C   = cp.Parameter(nonneg=True, name="n1C") # drawdown 한도
    x_min = cp.Parameter(name="x_min")
    x_max = cp.Parameter(name="x_max")

    # --- 목적함수: y_hat(N)^T x 최대화 ---
    # Y_hat[N-1] == y_hat(t=N) (0-indexed)
    objective = cp.Maximize(Y_hat[N - 1] @ x)

    # --- 제약식 ---
    constraints = []

    # u_0 = 0
    constraints.append(u[0] == 0)

    for k in range(1, N + 1):
        y_k = Y_hat[k - 1]          # y_hat(t=k), shape (m,)

        # drawdown 제약: u_k - y_k^T x <= n1C
        constraints.append(u[k] - y_k @ x <= n1C)

        # running max >= 현재 포트폴리오 cumulative return
        constraints.append(u[k] >= y_k @ x)

        # running max 단조증가
        constraints.append(u[k] >= u[k - 1])

    # box constraint
    constraints.append(x >= x_min)
    constraints.append(x <= x_max)

    # --- 문제 정의 ---
    problem = cp.Problem(objective, constraints)
    assert problem.is_dcp(), "Problem is not DCP!"

    # --- CvxpyLayer 생성 ---
    layer = CvxpyLayer(
        problem,
        parameters=[Y_hat, n1C, x_min, x_max],
        variables=[x, u],
    )
    return layer


def solve_portfolio(
    y_hat: torch.Tensor,
    opt_layer: CvxpyLayer,
    n1: float,
    C: float,
    x_min: float,
    x_max: float,
) -> torch.Tensor:
    """
    Optimization Layer를 통해 최적 포트폴리오 x*를 계산

    Args:
        y_hat     : (batch, N, m) - 예측 누적 수익률 경로
        opt_layer : CvxpyLayer (build_optimization_layer로 생성)
        n1        : drawdown 비율 한도
        C         : 자본 규모
        x_min     : 최소 비중
        x_max     : 최대 비중
    Returns:
        x_star : (batch, m) - 최적 포트폴리오 가중치
    """
    batch, N, m = y_hat.shape

    n1C_val   = torch.tensor(n1 * C, dtype=torch.float64)
    x_min_val = torch.tensor(x_min, dtype=torch.float64)
    x_max_val = torch.tensor(x_max, dtype=torch.float64)

    x_stars = []
    for b in range(batch):
        Y_hat_b = y_hat[b].double()  # (N, m)
        x_star_b, _ = opt_layer(
            Y_hat_b,
            n1C_val,
            x_min_val,
            x_max_val,
            solver_args={"solve_method": "ECOS"},
        )
        x_stars.append(x_star_b.float())

    x_star = torch.stack(x_stars, dim=0)  # (batch, m)
    return x_star


# =============================================================================
# Step 4. Realized Portfolio Path: (x_star, y_real) -> w_real(t)
# =============================================================================

def compute_realized_path(
    x_star: torch.Tensor,
    y_real: torch.Tensor,
) -> torch.Tensor:
    """
    실제 누적 수익률 경로 w_real(t) = x*(theta)^T y_real(t) 계산

    Args:
        x_star : (batch, m)    - 최적 포트폴리오 가중치
        y_real : (batch, N, m) - 실제 누적 수익률 경로
    Returns:
        w_real : (batch, N) - 포트폴리오 실현 누적 수익률 경로
    """
    # einsum: w_real[b,t] = sum_j x_star[b,j] * y_real[b,t,j]
    w_real = torch.einsum("bj, btj -> bt", x_star, y_real)  # (batch, N)
    return w_real


# =============================================================================
# Step 5. Performance Metrics: w_real -> R_real, M_real
# =============================================================================

def compute_return(
    w_real: torch.Tensor,
    d: float,
    C: float,
) -> torch.Tensor:
    """
    실현 수익률 R_real 계산
    R_real = (1 / dC) * w_real(N)

    Args:
        w_real : (batch, N)
        d      : 연수 (years in the time interval)
        C      : 자본 규모
    Returns:
        R_real : (batch,)
    """
    R_real = w_real[:, -1] / (d * C)
    return R_real


def compute_max_drawdown(w_real: torch.Tensor) -> torch.Tensor:
    """
    실현 MaxDD 계산 (differentiable 근사)

    D_real(t) = max_{tau <= t} w_real(tau) - w_real(t)
    M_real    = max_{1 <= t <= N} D_real(t)

    Note:
        torch.cummax는 미분 가능하지만 max는 argmax가 겹치면
        gradient가 불안정할 수 있음.
        DFL loss의 M_real 항은 gradient를 통해 theta를 업데이트하는 데 사용.

    Args:
        w_real : (batch, N)
    Returns:
        M_real : (batch,)
    """
    # running maximum up to each time step
    running_max, _ = torch.cummax(w_real, dim=1)  # (batch, N)

    # drawdown at each time step
    drawdown = running_max - w_real                # (batch, N)

    # maximum drawdown
    M_real = drawdown.max(dim=1).values            # (batch,)
    return M_real


# =============================================================================
# Step 6. DFL Loss Function
# =============================================================================

def dfl_loss(
    R_real: torch.Tensor,
    M_real: torch.Tensor,
    lam: float,
) -> torch.Tensor:
    """
    DFL Loss Function
    L(theta) = -R_real + lambda * M_real

    수익률 최대화 + MDD 최소화

    Args:
        R_real : (batch,) - 실현 수익률
        M_real : (batch,) - 실현 최대낙폭
        lam    : float    - MDD 패널티 가중치
    Returns:
        loss : scalar
    """
    loss = (-R_real + lam * M_real).mean()
    return loss


# =============================================================================
# Full Pipeline (End-to-End)
# =============================================================================

def forward_pass(
    z: torch.Tensor,
    r_real: torch.Tensor,
    pred_model: PredictionModel,
    opt_layer: CvxpyLayer,
    n1: float,
    C: float,
    d: float,
    x_min: float,
    x_max: float,
    lam: float,
) -> dict:
    """
    DFL 전체 파이프라인 (Forward Pass)

    Args:
        z          : (batch, input_dim)  - 입력 피처
        r_real     : (batch, N, m)       - 실현 수익률 (ground truth)
        pred_model : PredictionModel
        opt_layer  : CvxpyLayer
        n1         : drawdown 한도 비율
        C          : 자본 규모
        d          : 연수
        x_min      : 최소 비중
        x_max      : 최대 비중
        lam        : MDD 패널티 가중치
    
    Returns:
        dict with keys:
            r_hat   : (batch, N, m)  예측 수익률
            y_hat   : (batch, N, m)  예측 누적 수익률
            x_star  : (batch, m)     최적 포트폴리오
            y_real  : (batch, N, m)  실현 누적 수익률
            w_real  : (batch, N)     실현 포트폴리오 경로
            R_real  : (batch,)       실현 수익률
            M_real  : (batch,)       실현 MDD
            loss    : scalar         DFL Loss
    """
    # Step 1: z -> r_hat
    r_hat = pred_model(z)                           # (batch, N, m)

    # Step 2: r_hat -> y_hat
    y_hat = compute_cumulative_path(r_hat)          # (batch, N, m)

    # Step 3: y_hat -> x_star  (LP)
    x_star = solve_portfolio(
        y_hat, opt_layer, n1, C, x_min, x_max
    )                                               # (batch, m)

    # Step 4: x_star + y_real -> w_real
    y_real = compute_cumulative_path(r_real)        # (batch, N, m)
    w_real = compute_realized_path(x_star, y_real)  # (batch, N)

    # Step 5: w_real -> R_real, M_real
    R_real = compute_return(w_real, d, C)           # (batch,)
    M_real = compute_max_drawdown(w_real)           # (batch,)

    # Step 6: Loss
    loss = dfl_loss(R_real, M_real, lam)            # scalar

    return {
        "r_hat" : r_hat,
        "y_hat" : y_hat,
        "x_star": x_star,
        "y_real": y_real,
        "w_real": w_real,
        "R_real": R_real,
        "M_real": M_real,
        "loss"  : loss,
    }